<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/skill_check_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

GPU available: True
Mon Oct 12 17:01:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [2]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q "qwen-omni-utils[decord]" pymupdf python-docx sentence-transformers pandas scikit-learn
# If the Qwen2.5-Omni import fails, use the dev build instead:
# !pip install -q git+https://github.com/huggingface/transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 90.3 MB/s eta 0:00:00


In [1]:
import os, gc, glob, subprocess
import torch, pymupdf, docx
import numpy as np, pandas as pd
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor, BitsAndBytesConfig
from qwen_omni_utils import process_mm_info

assert torch.cuda.is_available(), "No GPU: set the runtime type to T4 GPU and restart."

model_id = "Qwen/Qwen2.5-Omni-3B"
processor = Qwen2_5OmniProcessor.from_pretrained(model_id)

def load(quantize_4bit=False):
    kwargs = dict(torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")
    if quantize_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4",
            llm_int8_skip_modules=["audio_tower", "visual", "lm_head"])
    return Qwen2_5OmniForConditionalGeneration.from_pretrained(model_id, **kwargs)

try:
    model = load(quantize_4bit=False); print("Loaded fp16")
except torch.OutOfMemoryError:
    gc.collect(); torch.cuda.empty_cache()
    model = load(quantize_4bit=True); print("fp16 didn't fit, loaded 4-bit")

if hasattr(model, "disable_talker"):      # text output only: free the speech modules
    model.disable_talker()
torch.cuda.empty_cache()
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

chat_template.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

tokenizer_config.json:   0%|          | 0.00/6.47k [00:00<?, ?B/s]

...

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

spk_dict.pt: reconstructing file:   0%|          |  0.00B /  260kB            

spk_dict.pt: downloading bytes:           |  0.00B            

Loaded fp16
GPU memory used: 9.4 GB


In [2]:
IMG = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}
VID = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
AUD = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}
TXT = {".txt", ".md", ".csv", ".json", ".html"}

MAX_CHARS, MAX_PIECES = 12000, 8      # long text is summarized piece by piece
VIDEO_CHUNK_SEC = 20                  # long videos are summarized clip by clip

SYSTEM = {"role": "system", "content": [{"type": "text",
    "text": "You are a concise assistant. Answer directly in plain text. Never ask the user questions."}]}
SUMMARY_PROMPT = "What is this about? Give a short summary of the main topic and key points in at most 4 sentences."

# Video quality tiers, tried in order if the GPU runs out of memory
VIDEO_SETTINGS = [
    {"nframes": 24, "max_pixels": 360 * 420},
    {"nframes": 12, "max_pixels": 224 * 224},
    {"nframes": 8,  "max_pixels": 168 * 168},
]

In [3]:
# ---- low-level: run the model on a list of content parts ----
def _run(conversation, max_new_tokens):
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)
    inputs = processor(text=text, audio=audios, images=images, videos=videos,
                       return_tensors="pt", padding=True, use_audio_in_video=False)
    inputs = inputs.to("cuda").to(torch.float16)
    with torch.inference_mode():
        out = model.thinker.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

def run_content(content, max_new_tokens=200):
    is_video = any(c["type"] == "video" for c in content)
    for tier in (VIDEO_SETTINGS if is_video else [None]):
        if tier:
            for c in content:
                if c["type"] == "video": c.update(tier)
        try:
            return _run([SYSTEM, {"role": "user", "content": content}], max_new_tokens)
        except torch.OutOfMemoryError:
            print("Out of memory, retrying with smaller settings...")
        finally:
            gc.collect(); torch.cuda.empty_cache()
    return "Still out of memory."

# ---- turn any input into model-ready content ----
def _text_content(text):
    if len(text) > MAX_CHARS:             # long text: summarize each piece, then the summaries
        pieces = [text[i:i + MAX_CHARS] for i in range(0, len(text), MAX_CHARS)][:MAX_PIECES]
        notes = [f"Part {i + 1}: " + run_content(
                    [{"type": "text", "text": f"{p}\n\nSummarize this part in 3 sentences."}], 120)
                 for i, p in enumerate(pieces)]
        text = "\n".join(notes)
    return [{"type": "text", "text": f"{text}\n\n{SUMMARY_PROMPT}"}]

def _pdf_content(path):
    with pymupdf.open(path) as doc:
        text = "\n".join(page.get_text() for page in doc)
        if len(text.strip()) > 200:
            return _text_content(text)
        content = []                       # scanned PDF: first 3 pages as images
        for i, page in enumerate(doc):
            if i >= 3: break
            png = f"/content/_page{i}.png"
            page.get_pixmap(dpi=100).save(png)
            content.append({"type": "image", "image": png})
    return content + [{"type": "text", "text": SUMMARY_PROMPT}]

def build_content(src):
    if not src.startswith("http") and not os.path.exists(src):
        return _text_content(src)          # plain text typed in directly
    ext = os.path.splitext(src.split("?")[0])[1].lower()
    if ext in IMG: return [{"type": "image", "image": src}, {"type": "text", "text": SUMMARY_PROMPT}]
    if ext in VID: return [{"type": "video", "video": src, **VIDEO_SETTINGS[0]}, {"type": "text", "text": SUMMARY_PROMPT}]
    if ext in AUD: return [{"type": "audio", "audio": src}, {"type": "text", "text": SUMMARY_PROMPT}]
    if ext == ".pdf": return _pdf_content(src)
    if ext == ".docx": return _text_content("\n".join(p.text for p in docx.Document(src).paragraphs))
    if ext in TXT:
        with open(src, errors="ignore") as f: return _text_content(f.read())
    raise ValueError(f"Unsupported file type: {ext}")

In [4]:
# ---- long videos: split into clips, describe each, then summarize the notes ----
def video_duration(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=nw=1:nk=1", path], capture_output=True, text=True)
    return float(r.stdout.strip())

def split_video(path, chunk_sec=VIDEO_CHUNK_SEC, out_dir="/content/chunks"):
    os.makedirs(out_dir, exist_ok=True)
    for f in glob.glob(f"{out_dir}/*.mp4"): os.remove(f)
    # "-c copy" = no re-encoding (seconds, not minutes). If a chunk fails to decode, use
    # ["-c:v", "libx264", "-preset", "veryfast"] instead of ["-c", "copy"].
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-an", "-c", "copy",
                    "-f", "segment", "-segment_time", str(chunk_sec), "-reset_timestamps", "1",
                    f"{out_dir}/chunk_%03d.mp4"], check=True)
    return sorted(glob.glob(f"{out_dir}/*.mp4"))

def summarize_video(src):
    if video_duration(src) <= VIDEO_CHUNK_SEC * 1.5:
        return run_content(build_content(src))
    notes = []
    for i, chunk in enumerate(split_video(src)):
        content = [{"type": "video", "video": chunk},
                   {"type": "text", "text": "Describe what happens in this clip in 2 sentences."}]
        notes.append(f"Part {i + 1}: {run_content(content, 80)}")
    prompt = "These are notes on consecutive parts of one video:\n\n" + "\n".join(notes) + "\n\n" + SUMMARY_PROMPT
    return run_content([{"type": "text", "text": prompt}], 200)

def summarize(src):
    """Any input -> summary text."""
    ext = os.path.splitext(src.split("?")[0])[1].lower() if os.path.exists(src) else ""
    return summarize_video(src) if ext in VID else run_content(build_content(src))

In [5]:
from sentence_transformers import SentenceTransformer, util

JUDGE_THRESHOLD = 0.5      # a skill is matched when judge >= this. Tune it in section 5.

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")  # CPU: keeps GPU free

tok = processor.tokenizer
def _ids(words): return sorted({tok.encode(w, add_special_tokens=False)[0] for w in words})
YES_IDS, NO_IDS = _ids(["YES", "Yes", "yes"]), _ids(["NO", "No", "no"])

JUDGE_SYSTEM = {"role": "system", "content": [{"type": "text",
    "text": "You are a strict classifier. Reply with exactly one word: YES or NO."}]}
LABEL_EXAMPLES = "'Programming' (not 'Tower of Hanoi puzzle'), 'Chess' (not 'two players moving pieces'), 'Cooking', 'Dogs', 'Finance', 'Movies'"

def skill_text(skill): return f"{skill['name']}: {skill['description']}"

def clean_label(raw, max_words=3):
    label = raw.strip().split("\n")[0].strip(" \"'`*.:;-")
    if label.lower().startswith("label"):
        label = label.split(":", 1)[-1].strip(" \"'`*.")
    return " ".join(label.split()[:max_words])

def make_label(summary):
    """1 to 2 word broad topic (the field, not a description), e.g. 'Programming'."""
    q = (f"Content summary:\n{summary}\n\n"
         "What general subject or field is this content about? Answer with 1 or 2 words only: the broad "
         f"category, not a description or specific details. Examples: {LABEL_EXAMPLES}. "
         "Output only the category.")
    return clean_label(run_content([{"type": "text", "text": q}], max_new_tokens=8), max_words=3)

def judge_skill(label, summary, skill):
    """P(YES) that the content is related to the skill, from next-token logits."""
    q = (f"Content topic: {label}\nContent summary: {summary}\n\nSkill: {skill_text(skill)}\n\n"
         "Is the content clearly about this skill's subject area, so that a person with this skill "
         "would find it relevant? Answer YES or NO.")
    conv = [JUDGE_SYSTEM, {"role": "user", "content": [{"type": "text", "text": q}]}]
    text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=text, return_tensors="pt", padding=True, use_audio_in_video=False)
    inputs = inputs.to("cuda").to(torch.float16)
    with torch.inference_mode():
        out = model.thinker.generate(**inputs, max_new_tokens=1, do_sample=False,
                                     output_scores=True, return_dict_in_generate=True)
    logits = out.scores[0][0].float()
    return torch.sigmoid(torch.logsumexp(logits[YES_IDS], 0) - torch.logsumexp(logits[NO_IDS], 0)).item()

def score_skills(label, summary, skills):
    emb_s = embedder.encode(summary, convert_to_tensor=True, normalize_embeddings=True)
    emb_k = embedder.encode([skill_text(s) for s in skills], convert_to_tensor=True, normalize_embeddings=True)
    sims = util.cos_sim(emb_s, emb_k)[0].tolist()
    rows = [{"skill": s["name"], "judge": round(judge_skill(label, summary, s), 3), "embed": round(sim, 3)}
            for s, sim in zip(skills, sims)]
    gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(rows).sort_values("judge", ascending=False).reset_index(drop=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
def check(src, skills, threshold=None):
    """
    src    : text, or a path/URL to an image, video, audio, pdf, docx or text file
    skills : list of {"name": ..., "description": ...}
    returns: dict with
        related  -> True if the content matches at least one skill
        matches  -> names of matched skills
        label    -> short topic ("Programming")
        summary  -> full generated summary
        table    -> per-skill scores
    """
    threshold = JUDGE_THRESHOLD if threshold is None else threshold
    summary = summarize(src)
    label = make_label(summary)
    table = score_skills(label, summary, skills)
    table["matched"] = table.judge >= threshold
    matches = table[table.matched].skill.tolist()
    return {"related": bool(matches), "matches": matches, "label": label, "summary": summary, "table": table}

def show(r, verbose=False):
    print(f"Topic   : {r['label']}")
    print(f"Related : {r['related']}   Matched: {r['matches']}")
    if verbose: print(f"Summary : {r['summary']}")      # the summary is only used internally
    return r["table"]

In [7]:
# Test skills: what a user would define when creating a profile
TEST_SKILLS = [
    {"name": "Python Programming",  "description": "writing, debugging and explaining Python code, scripts and software"},
    {"name": "Cooking",             "description": "recipes, ingredients, food preparation and kitchen techniques"},
    {"name": "Pets & Animals",      "description": "dogs, cats, wildlife, animal behavior and pet care"},
    {"name": "Finance & Investing", "description": "stock markets, investing, banking, budgeting and economics"},
    {"name": "Movies & Cinema",     "description": "films, scenes, actors, directing and film analysis"},
    {"name": "Biology & Science",   "description": "biology, plants, chemistry, physics and natural science"},
]

In [8]:
# Try it: text
show(check("Today we are going to make pizza together. First, stretch the dough and add tomato sauce.", TEST_SKILLS))

Topic   : Cooking
Related : True   Matched: ['Cooking']


,skill,judge,embed,matched
0,Cooking,0.890,0.351,True
1,Biology & Science,0.239,0.194,False
2,Movies & Cinema,0.143,0.218,False
3,Python Programming,0.106,0.244,False
4,Pets & Animals,0.004,0.046,False
5,Finance & Investing,0.003,0.035,False


In [9]:
# Try it: your own files (edit the paths)
for path in ["/content/30sec2.mp4", "/content/programming.mp4", "/content/9_test3.jpeg","/content/7_test1.png"]:
    print("\n" + path); display(show(check(path, TEST_SKILLS)))


/content/30sec2.mp4


qwen-vl-utils using torchcodec to read video.
[transformers] Qwen2VL video processing does not apply the per-frame pixel cap the reference implementation (qwen-vl-utils) applies, so some videos cost far more tokens than they would there. In v5.22 the capped behavior will become the default and `cap_pixels_per_frame` will be removed. Pass `cap_pixels_per_frame=True` to adopt the reference behavior now, or `False` to keep the current behavior and silence this warning.


Topic   : Zoology
Related : True   Matched: ['Pets & Animals', 'Biology & Science', 'Movies & Cinema']


,skill,judge,embed,matched
0,Pets & Animals,0.711,0.331,True
1,Biology & Science,0.655,0.105,True
2,Movies & Cinema,0.551,0.075,True
3,Python Programming,0.071,0.099,False
4,Cooking,0.017,0.036,False
5,Finance & Investing,0.003,-0.005,False



/content/programming.mp4
Topic   : Programming
Related : True   Matched: ['Python Programming']


,skill,judge,embed,matched
0,Python Programming,0.890,0.132,True
1,Movies & Cinema,0.109,0.039,False
2,Cooking,0.008,0.004,False
3,Biology & Science,0.008,0.002,False
4,Finance & Investing,0.004,-0.128,False
5,Pets & Animals,0.004,-0.002,False



/content/9_test3.jpeg
Topic   : People
Related : False   Matched: []


,skill,judge,embed,matched
0,Movies & Cinema,0.454,0.133,False
1,Python Programming,0.301,-0.020,False
2,Biology & Science,0.044,0.003,False
3,Cooking,0.011,-0.015,False
4,Pets & Animals,0.007,0.088,False
5,Finance & Investing,0.005,-0.049,False



/content/7_test1.png
Topic   : Illustration
Related : False   Matched: []


,skill,judge,embed,matched
0,Movies & Cinema,0.335,0.112,False
1,Python Programming,0.276,0.067,False
2,Cooking,0.163,0.235,False
3,Pets & Animals,0.130,0.166,False
4,Biology & Science,0.029,0.111,False
5,Finance & Investing,0.004,0.004,False
